# Type indexer integration test

This notebook follows the repository's actual pipeline:

1. Open a `DotVectorDB` database.
2. Ingest supported documents with `DotVectorDB.ingest_documents()`.
3. Construct the current `TyoeIndexer` with the same dependencies used by the other indexers.
4. Run `extract_tags_from_ingested_chunks()`.
5. Inspect the resulting tags, chunk flags, and LLM token usage.

The implementation currently spells the class `TyoeIndexer`, so the notebook imports that exact name. Running ingestion and indexing can make paid embedding and LLM API requests.

## Imports and environment

In [ ]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

working_dir = Path.cwd().resolve()
repo_root = next(
    (candidate for candidate in (working_dir, *working_dir.parents)
     if (candidate / 'dotvectordb' / 'graph' / 'type_indexer.py').is_file()),
    None,
)
assert repo_root is not None, 'Could not find the DotVectorDB repository root.'
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

load_dotenv(repo_root / '.env')

from dotvectordb import DotVectorDB
from dotvectordb.graph.type_indexer import TyoeIndexer

print(f'Repository: {repo_root}')

## Configuration

Set `DOCUMENT_PATHS` to individual files and/or directories. Directory traversal is recursive. Supported formats are `.txt`, `.pdf`, `.docx`, `.xlsx`, `.xlsm`, and `.pptx`.

Use a fresh `DATABASE_PATH` when you want a completely independent run. Existing databases are resumed rather than erased.

In [ ]:
DOCUMENT_PATHS = [
    './documents',  # Replace with your document file or directory.
]
DATABASE_PATH = './type_indexer_test.db'
REQUESTS_PER_MINUTE = 1000
ASYNC_COOLDOWN_SECONDS = 60
TOKENIZER_NAME = os.getenv('TOKENIZER', 'gpt-4o')

resolved_document_paths = [
    Path(path).expanduser().resolve() for path in DOCUMENT_PATHS
]
missing_paths = [str(path) for path in resolved_document_paths if not path.exists()]
assert not missing_paths, 'Document paths do not exist: ' + ', '.join(missing_paths)
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY is missing from the environment or .env.'

print('Documents:')
for path in resolved_document_paths:
    print(f'  - {path}')
print(f'Database: {Path(DATABASE_PATH).expanduser().resolve()}')

## Open the database

In [ ]:
database = DotVectorDB(
    path=DATABASE_PATH,
    tokenizer_name=TOKENIZER_NAME,
    verbose=True,
)
await database.open()
print('Database is open.')

## Ingest and embed the documents

This calls the repository's public ingestion function. It extracts text, chunks it, stores the chunks, and embeds every unprocessed chunk.

In [ ]:
await database.ingest_documents(
    input_document_list=[str(path) for path in resolved_document_paths],
    requests_per_minute=REQUESTS_PER_MINUTE,
    async_cooldown=ASYNC_COOLDOWN_SECONDS,
)
print('Document ingestion finished.')

## Verify the ingested chunks before indexing

In [ ]:
cursor = await database.conn.execute(
    '''
    SELECT chunk_id, source_id, is_embedded, are_tags_discovered,
           substr(content, 1, 160) AS preview
    FROM chunk
    ORDER BY rowid
    '''
)
chunk_rows = await cursor.fetchall()
await cursor.close()

assert chunk_rows, 'No chunks were ingested.'
print(f'Chunks in database: {len(chunk_rows)}')
for chunk_id, source_id, is_embedded, tags_done, preview in chunk_rows[:10]:
    print({
        'chunk_id': chunk_id,
        'source_id': source_id,
        'is_embedded': bool(is_embedded),
        'are_tags_discovered': bool(tags_done),
        'preview': preview,
    })

## Construct and run the type indexer

These dependencies come directly from the open `DotVectorDB` instance. The language-model dependency is its async `generate` method, matching the pattern used for `EntityIndexer` and `RelationshipIndexer`.

In [ ]:
type_indexer = TyoeIndexer(
    conn=database.conn,
    lock=database.lock,
    language_model=database.language_model.generate,
    embedding_model=database.embedding_model,
    hash_func=database.hashing_func.content_hash,
    tag_func=database.add_entity_tag,
)

await type_indexer.extract_tags_from_ingested_chunks(
    requests_per_minute=REQUESTS_PER_MINUTE,
    async_cooldown=ASYNC_COOLDOWN_SECONDS,
    verbose=True,
)
print('Type-indexing call finished.')

## Inspect results

In [ ]:
tag_cursor = await database.conn.execute(
    'SELECT entity_tag, num_of_instances FROM entity_tags ORDER BY entity_tag'
)
tags = await tag_cursor.fetchall()
await tag_cursor.close()

status_cursor = await database.conn.execute(
    '''
    SELECT
        COUNT(*) AS total_chunks,
        SUM(CASE WHEN are_tags_discovered = 1 THEN 1 ELSE 0 END) AS processed_chunks
    FROM chunk
    '''
)
total_chunks, processed_chunks = await status_cursor.fetchone()
await status_cursor.close()

token_cursor = await database.conn.execute(
    "SELECT COALESCE(SUM(total_tokens), 0) FROM llm_token_usage WHERE stage = 'type_discovery'"
)
type_discovery_tokens = (await token_cursor.fetchone())[0]
await token_cursor.close()

print(f'Processed chunks: {processed_chunks}/{total_chunks}')
print(f'Type-discovery tokens: {type_discovery_tokens}')
print(f'Discovered tags: {len(tags)}')
for tag, instances in tags:
    print(f'  - {tag} ({instances} instances)')

## Close the database

Run this cell even if an earlier indexing cell raises an exception.

In [ ]:
await database.close()
print('Database closed.')